# Dynamic Light Scattering, Zeta Potential, and Growth Inhibition Measurements for Self-Associating Amphiphiles Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook follows a step-by-step workflow:
- load the Croissant metadata,
- inspect dataset-level information,
- discover available record sets and fields using their `@id` values,
- load one or more record sets into pandas DataFrames,
- perform basic data cleaning and exploratory analysis,
- create visualizations when suitable data are available.

> Important: all dataset entities should be referenced by their `@id` values. Because this dataset may evolve over time, the notebook uses variables and metadata inspection to dynamically identify available record sets and fields.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

In this section we create a dataset object from the Croissant JSON-LD URL and inspect high-level metadata. We keep the metadata object as a single object and use attribute access where possible.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.tnk3-xhn0/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Identifier:', getattr(metadata, 'identifier', 'N/A'))
print('Version:', getattr(metadata, 'version', 'N/A'))
print('License:', getattr(metadata, 'license', 'N/A'))

In [ ]:
# Optional: inspect a JSON representation of the metadata object
# This can be useful for understanding the schema structure.

metadata_json = metadata.to_json()
print(type(metadata_json))
print('Top-level keys preview:')
print(list(metadata_json.keys())[:20])

## 2. Data Overview
Review available record sets, fields, and their IDs.

Because record sets may differ across Croissant datasets, we inspect the metadata dynamically. The key goal is to capture the `@id` of each record set and field, then use those IDs in later steps.

In [ ]:
def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

record_sets_meta = metadata_json.get('recordSet', [])
print(f'Number of record sets declared in metadata: {len(record_sets_meta)}')

record_set_summary = []
for rs in record_sets_meta:
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', 'N/A')
    fields = as_list(rs.get('field', []))
    record_set_summary.append({
        'record_set_id': rs_id,
        'record_set_name': rs_name,
        'field_count': len(fields)
    })

record_set_summary_df = pd.DataFrame(record_set_summary)
record_set_summary_df

In [ ]:
# Display each record set and its fields using @id references
for rs in record_sets_meta:
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', 'N/A')
    print('\n' + '=' * 100)
    print(f'Record set @id: {rs_id}')
    print(f'Record set name: {rs_name}')
    print('-' * 100)
    
    fields = as_list(rs.get('field', []))
    if not fields:
        print('No fields declared for this record set.')
        continue
    
    for field in fields:
        field_id = field.get('@id', 'N/A')
        field_name = field.get('name', 'N/A')
        field_description = field.get('description', 'N/A')
        field_data_type = field.get('dataType', 'N/A')
        source = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        column_ref = source.get('column', 'N/A')
        file_ref = source.get('fileObject', 'N/A')
        print(f'  Field @id: {field_id}')
        print(f'    name: {field_name}')
        print(f'    dataType: {field_data_type}')
        print(f'    source.column: {column_ref}')
        print(f'    source.fileObject: {file_ref}')
        print(f'    description: {field_description}')

If record sets are available, we can also preview raw records directly with `dataset.records(record_set=<record_set_id>)`.

The next cell automatically selects the first available record set `@id` for a small preview.

In [ ]:
available_record_set_ids = [rs.get('@id') for rs in record_sets_meta if rs.get('@id')]

if available_record_set_ids:
    preview_record_set_id = available_record_set_ids[0]
    print('Previewing records from record set @id:', preview_record_set_id)
    
    try:
        for i, x in enumerate(dataset.records(record_set=preview_record_set_id)):
            print(x)
            if i >= 2:
                break
    except Exception as e:
        print('Could not preview records for this record set.')
        print(type(e).__name__, str(e))
else:
    print('No record sets were declared in the metadata. This may indicate that the dataset is metadata-rich but does not currently expose tabular record sets through Croissant.')

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis.

This section uses `@id` values for record sets and stores each extracted table in a dictionary keyed by the record set `@id`.

If the dataset does not expose record sets, the notebook will safely continue and use metadata-level information for demonstration.

In [ ]:
# Extract data from each available record set
record_set_ids = available_record_set_ids
dataframes = {}
load_errors = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Loaded {len(df)} records from {record_set_id}')
    except Exception as e:
        load_errors[record_set_id] = f'{type(e).__name__}: {str(e)}'
        print(f'Failed to load records from {record_set_id}: {load_errors[record_set_id]}')

print('\nLoaded record sets:', list(dataframes.keys()))
print('Load errors:', load_errors if load_errors else 'None')

In [ ]:
# Inspect the first successfully loaded DataFrame, if any
if dataframes:
    example_record_set_id = next(iter(dataframes.keys()))
    print('Example record set @id:', example_record_set_id)
    print('Columns:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No DataFrames were loaded from record sets.')

To support reproducible workflows, it is useful to create helper structures that map each record set `@id` to its field `@id` values.

In [ ]:
record_set_to_fields = {}

for rs in record_sets_meta:
    rs_id = rs.get('@id')
    fields = as_list(rs.get('field', []))
    record_set_to_fields[rs_id] = [field.get('@id') for field in fields if field.get('@id')]

field_map_preview = pd.DataFrame([
    {'record_set_id': rs_id, 'field_ids': field_ids}
    for rs_id, field_ids in record_set_to_fields.items()
])
field_map_preview

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section is written to adapt to whichever record sets are available. It automatically:
- finds a loaded DataFrame,
- detects numeric columns,
- filters rows above a threshold,
- standardizes a numeric field,
- optionally groups by a categorical column.

When no record-set data are available, we fall back to a metadata-level exploratory example using dataset keywords and declared distributions.

In [ ]:
selected_record_set_id = None
selected_df = None
numeric_field_id = None
group_field_id = None

for rs_id, df in dataframes.items():
    if df.empty:
        continue
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        selected_record_set_id = rs_id
        selected_df = df.copy()
        numeric_field_id = numeric_candidates[0]
        non_numeric_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == 'object']
        group_field_id = non_numeric_candidates[0] if non_numeric_candidates else None
        break

if selected_df is not None:
    print('Selected record set @id:', selected_record_set_id)
    print('Selected numeric field @id/column:', numeric_field_id)
    print('Selected grouping field @id/column:', group_field_id)
    
    threshold = selected_df[numeric_field_id].median()
    filtered_df = selected_df[selected_df[numeric_field_id] > threshold].copy()
    print(f'Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows')
    display(filtered_df.head())
    
    std_value = filtered_df[numeric_field_id].std()
    if pd.notna(std_value) and std_value != 0:
        normalized_col = f'{numeric_field_id}_normalized'
        filtered_df[normalized_col] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / std_value
        )
        print(f'Normalized {numeric_field_id} for filtered records:')
        display(filtered_df[[numeric_field_id, normalized_col]].head())
    else:
        print(f'Skipping normalization because {numeric_field_id} has zero or undefined standard deviation in the filtered data.')
    
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=True).reset_index()
        print(f'Grouped mean of {numeric_field_id} by {group_field_id}:')
        display(grouped_df.head())
else:
    print('No suitable numeric record-set data were available for DataFrame-based EDA.')
    print('Proceeding with metadata-level exploration instead.')

In [ ]:
# Metadata-level EDA fallback: explore keywords and distributions
metadata_keywords = metadata_json.get('keywords', [])
metadata_distributions = metadata_json.get('distribution', [])

keywords_df = pd.DataFrame({'keyword': metadata_keywords})
keywords_df['keyword_length'] = keywords_df['keyword'].astype(str).str.len()

distribution_df = pd.DataFrame(metadata_distributions)
if '@id' in distribution_df.columns:
    distribution_df['distribution_id_length'] = distribution_df['@id'].astype(str).str.len()

print('Keywords table:')
display(keywords_df)

print('Distributions table:')
display(distribution_df.head())

As an additional structured summary, the next cell turns selected metadata elements into analysis-ready tables.

In [ ]:
dataset_profile = pd.DataFrame([
    {
        'dataset_id': metadata_json.get('@id', 'N/A'),
        'dataset_name': metadata_json.get('name', 'N/A'),
        'identifier': metadata_json.get('identifier', 'N/A'),
        'version': metadata_json.get('version', 'N/A'),
        'keyword_count': len(metadata_json.get('keywords', [])),
        'distribution_count': len(metadata_json.get('distribution', [])),
        'record_set_count': len(metadata_json.get('recordSet', [])),
        'author_count': len(metadata_json.get('author', [])),
        'funding_count': len(metadata_json.get('funding', []))
    }
])

dataset_profile

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The notebook first attempts to visualize a numeric field from a loaded record set. If no tabular records are available, it creates metadata-based visualizations instead.

In [ ]:
if selected_df is not None and numeric_field_id is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.histplot(selected_df[numeric_field_id].dropna(), kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title(f'Distribution of {numeric_field_id}')
    axes[0].set_xlabel(numeric_field_id)
    
    sns.boxplot(x=selected_df[numeric_field_id], ax=axes[1], color='orange')
    axes[1].set_title(f'Boxplot of {numeric_field_id}')
    axes[1].set_xlabel(numeric_field_id)
    
    plt.tight_layout()
    plt.show()
    
    if group_field_id and group_field_id in selected_df.columns:
        top_groups = (
            selected_df[group_field_id]
            .astype(str)
            .value_counts()
            .head(10)
            .index
        )
        plot_df = selected_df[selected_df[group_field_id].astype(str).isin(top_groups)].copy()
        
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=plot_df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.title(f'{numeric_field_id} by {group_field_id} (top groups)')
        plt.tight_layout()
        plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    if not keywords_df.empty:
        sns.barplot(data=keywords_df.sort_values('keyword_length', ascending=False), x='keyword_length', y='keyword', ax=axes[0], palette='viridis')
        axes[0].set_title('Keyword lengths')
        axes[0].set_xlabel('Character count')
        axes[0].set_ylabel('Keyword')
    else:
        axes[0].text(0.5, 0.5, 'No keywords available', ha='center', va='center')
        axes[0].set_axis_off()
    
    if not distribution_df.empty and '@id' in distribution_df.columns:
        sns.histplot(distribution_df['distribution_id_length'].dropna(), kde=False, ax=axes[1], color='purple')
        axes[1].set_title('Distribution @id length distribution')
        axes[1].set_xlabel('Character count')
    else:
        axes[1].text(0.5, 0.5, 'No distributions available', ha='center', va='center')
        axes[1].set_axis_off()
    
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:

- loaded the dataset from its Croissant schema URL using `mlcroissant`,
- inspected dataset-level metadata such as name, description, identifier, version, and license,
- dynamically searched for record sets and fields using their `@id` values,
- attempted to extract available record sets into pandas DataFrames,
- applied an adaptive EDA workflow that uses record-level data when available,
- provided a metadata-level fallback analysis when record sets are absent or unavailable.

For this FAIR^2 dataset, the metadata describes a rich experimental resource related to dynamic light scattering, zeta potential, and growth inhibition measurements for self-associating amphiphiles. If future versions of the Croissant file expose record sets directly, the same notebook structure can immediately support deeper tabular analysis using the corresponding record set and field `@id` values.